# Example 16.1 / 1: Solve and Count Boards With alldiff

Companion notebook for **16.1 N-Queens** in the AMPL lab Documentation, section "Solve and Count Boards With `alldiff`". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Solve and Count Boards With `alldiff`

`alldiff {j in COLS} expr` requires the listed expressions to take pairwise different values; `alldiff` and CP-Style Models (AMPL lab documentation, section 7.6.1) explains it in full. One `alldiff` per attack direction is the whole model. This is a feasibility problem: there is no objective, because every board that satisfies the rules is an answer.

The run solves an 8 × 8 board with HiGHS, checks `ampl.solve_result`, reads `Row` with `to_dict()`, and then checks the board in Python without reusing the model's expressions: for every pair of columns $j<k$, the rows must differ and `abs(rows[j] - rows[k])` must differ from `k - j`.

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
param n integer > 0;  # board size and number of queens
set COLS := 1..n;  # board columns, one queen in each
var Row {COLS} integer >= 1, <= n;  # row of the queen in column j

subject to SeparateRows:  # no two queens share a row
    alldiff {j in COLS} Row[j];
subject to SeparateDownDiagonals:  # no two queens share a row+column diagonal
    alldiff {j in COLS} Row[j] + j;
subject to SeparateUpDiagonals:  # no two queens share a row-column diagonal
    alldiff {j in COLS} Row[j] - j;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.param["n"] = 8

In [5]:
# run.py
ampl.solve(solver="highs", verbose=False)
print(ampl.solve_result)
rows = {j: round(r) for j, r in ampl.var["Row"].to_dict().items()}
for r in range(1, 9):
    print(" ".join("Q" if rows[j] == r else "." for j in range(1, 9)))
safe = all(
    rows[j] != rows[k] and abs(rows[j] - rows[k]) != k - j
    for j in range(1, 9)
    for k in range(j + 1, 9)
)
print("independent check:", safe)

solved
. . . Q . . . .
. . . . . . Q .
. . . . Q . . .
. Q . . . . . .
. . . . . Q . .
Q . . . . . . .
. . Q . . . . .
. . . . . . . Q
independent check: True


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
param n := 8;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs, solver_msg 0;
solve;
display solve_result, Row;

solve_result = solved

Row [*] :=
1  6
2  4
3  7
4  1
5  3
6  5
7  2
8  8
;



### Another way in amplpy

A `solved` status proves that this board exists; it says nothing about whether other boards exist. To find another one, add a **no-good cut**: a constraint that removes exactly one known solution and nothing else. Here it requires the new board to differ from a recorded board in at least one column. Repeating solve, record and exclude until the model is `infeasible` lists every board, and that final `infeasible` status is what proves none is left. Any other final status, such as `failure` or a time limit, proves neither.

The added lines below record each board in `Found` and exclude it with `Exclude`. `first_row_max` adds an optional restriction `Row[1] <= first_row_max`, used in the next section. The run adds them with `ampl.eval`, because amplpy has no method that declares new model components, then counts the boards for $n=1,\ldots,6$ and once more for $n=5$ with `first_row_max = 3`:

In [9]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [10]:
# data.py
ampl.param["n"] = 8

In [11]:
# run.py
ampl.eval("""
param first_row_max integer >= 1 default n;  # highest row allowed for the first queen
param nfound integer >= 0 default 0;  # solutions recorded so far
param Found {1..nfound, COLS};  # rows of each recorded solution
subject to FirstQueenHalf:  # optional restriction on the first queen
    Row[1] <= first_row_max;
subject to Exclude {s in 1..nfound}:  # no-good cut: differ from every recorded board
    exists {j in COLS} Row[j] != Found[s,j];
""")


def count_solutions(n, first_row_max):
    ampl.param["n"] = n
    ampl.param["first_row_max"] = first_row_max
    ampl.param["nfound"] = 0
    ampl.eval("reset data Found;")
    while True:
        ampl.solve(solver="highs", verbose=False)
        if ampl.solve_result != "solved":
            return int(ampl.param["nfound"].value()), ampl.solve_result
        rows = ampl.var["Row"].to_dict()  # read before nfound changes the model
        k = int(ampl.param["nfound"].value()) + 1
        ampl.param["nfound"] = k
        for j, r in rows.items():
            ampl.param["Found"][k, j] = round(r)


for n, m in [(1, 1), (2, 2), (3, 3), (4, 4), (5, 5), (6, 6), (5, 3)]:
    count, status = count_solutions(n, m)
    print(f"n = {n}, first_row_max = {m}: {count} solutions, final status {status}")

n = 1, first_row_max = 1: 1 solutions, final status infeasible
n = 2, first_row_max = 2: 0 solutions, final status failure
n = 3, first_row_max = 3: 0 solutions, final status infeasible
n = 4, first_row_max = 4: 2 solutions, final status infeasible
n = 5, first_row_max = 5: 10 solutions, final status infeasible
n = 6, first_row_max = 6: 4 solutions, final status infeasible
n = 5, first_row_max = 3: 6 solutions, final status infeasible


### Pure AMPL: data.dat and model.run

The same count for $n=5$ in pure AMPL, with the same `model.mod`; a `repeat` loop stops at the first status other than `solved`:

In [12]:
%%writefile data_2.dat
param n := 5;

Writing data_2.dat


In [13]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data_2.dat;

In [14]:
%%ampl_eval
# model.run
param first_row_max integer >= 1 default n;  # highest row allowed for the first queen
param nfound integer >= 0 default 0;  # solutions recorded so far
param Found {1..nfound, COLS};  # rows of each recorded solution
subject to FirstQueenHalf:  # optional restriction on the first queen
    Row[1] <= first_row_max;
subject to Exclude {s in 1..nfound}:  # no-good cut: differ from every recorded board
    exists {j in COLS} Row[j] != Found[s,j];

option solver highs, solver_msg 0;
repeat {
    solve;
    if solve_result <> "solved" then break;
    let nfound := nfound + 1;
    let {j in COLS} Found[nfound,j] := round(Row[j]);
};
printf "%d solutions, final status %s\n", nfound, solve_result;

10 solutions, final status infeasible
